# Research Diary 3

## 1

I had a working grasp of SAL (Sampled Anti-Aliased Likelihood): sample a continuous nuisance group $G$, locally average around each sampled transform to reduce aliasing, then pool (often max) across samples to approximate the profile likelihood. I understood the distinction between profile vs. marginal likelihoods, the roles of invariance/insensitivity/selectivity, and how SIFT/DSP-SIFT combine canonization and pooling. I also connected this to CNNs: receptive fields $V_j$, pushing transforms to the input for equivariance, and reading conv+pool as local likelihood + pooling.

### What I planned for this week:
* Dive deeper into Section 4 (Deep Convolutional Architectures) to connect minimally sufficient representations to CNN layers and spot optimization opportunities.
  * I succesfully did this and have a better understanding of how minimally sufficient representations transfer into CNN computation
* Explore what a minimally sufficient representation would look like if reconstruction is required (i.e., can we “add back” nuisance variables to render an image).
  * I was unable to get to this yet, but this is a direction I am interested in exploring
* Decide on a research direction and begin preliminary reading/experiments.
  * I have identified 2 possible directions for further research - diving more into transformations or looking at using this representation for image reconstruction

## 2

Papers:
* Visual Representations: Defining Properties and Deep Approximations, S. Soatto, A. Chiuso, ICLR 2016; [Web Link](https://arxiv.org/pdf/1411.7676v9), [Github Link](../Papers/Visual_Representations_defining_properties_and_deep_approximations.pdf)
  * Time Spent: 6 hrs
* [Link](https://chatgpt.com/share/68f2fb47-1188-8010-93d4-de990ad9d040) to AI Transcript
* Research Diary
  * Time Spent: 0.5 hrs

## 3

* Deformable templates for categories (3.2): Instead of placing a prior on the infinite-dimensional scene $\theta$ (shapes/reflectance/illumination), I can push intra-class variability into image-space deformations: pick a template $xk$ and explain any datum $y$ in the class via $y=g_k^{−1}x_k$. Practically this happens locally over receptive fields $V_j$, where small deformations are well-approximated and occlusion/clutter don’t dominate.
* Delta trick & alignment: Writing $p(y∣x_k,g_k)=​\delta(y-g_k^{-1}x_k)$ encodes that $y$ is exactly the warped template; integrating out $x_k$ yields scores of the form $p(g_k y∣ \theta_k)$. The inverse vs. forward action is a frame-of-reference choice (generative vs. alignment).
* Deep architectures as layered nuisance marginalization (4.x): A big deformation $g$ can be composed from small ones $(g=g_1 \circ g_2 \circ ...)$. Introducing separator variables $\theta_1, \theta_2, ...$ turns the single hard marginal into layer-wise marginals. Choosing $\theta_1 = g_2 \theta$ gives the neat Markov chain $\theta$ &rarr; $\theta_1$ &rarr; $y$ so the lower layer pools over data-space small transforms $(g_1)$ and the upper layer pools over representation-space small transforms ($g_2$). Discretizing $\theta_1$ into $K_1$ codes (filters) and sampling $g_1$ across $L_1$ actions (e.g., all spatial positions) produces the familiar $N×M×K_1$ feature tensor; stacking repeats the pattern.
* “Codes” vs. “classes”: Codes are layer-internal prototypes (edges/parts/textures), not semantic categories. Multiple codes can fire on the same patch; class decisions emerge after aggregating pooled evidence across layers.
* Not bag-of-words: I shouldn’t factor patch likelihoods independently because the joint $dP_G(\{g_j\}∣\theta)$ couples part configurations. The layered construction preserves dependencies while keeping computation tractable.
* Discussion takeaway: CNNs approximate hierarchical SAL: each layer marginalizes small local deformations; depth composes these to handle large/global nuisances. This view also explains why first-layer conv+nonlinearity with pooling resembles contrast-/rotation-aware descriptors under appropriate normalizations.

## 4

* How does kernel size impact the transformations that take place after using SAL marginalization.
  * I would like to do some numerical computation for 1x1, 3x3 and 5x5 kernel sizes
* “Adding back” nuisances for regeneration: Since invariants marginalize $g$, reconstruction needs either a learned posterior over $g$ or an auxiliary estimator.
* How does the transformations work when it comes to residual block connections in ResNet?
  * In particular I would like to see how the transformation would be when the end of the residual block is merged with the input.

## 5

N/A

## 6

* Change-of-variables with $\delta$: I understand the idea but need to be vigilant about why $\int f(x)\delta(y-g^{-1}x)dx = f(gy)$ (solve $x=gy$, consider Jacobians if not volume-preserving).
* Roles of $g_i, g_j, g_k$: I can mix them up unless I explicitly note which space each acts on (global alignment vs. receptive-field placement vs. intra-class deformation).
* Separator choice & order ($\theta_1 = g_2\theta$ vs. $\theta_1 = g_1\theta$): I see why the paper’s choice yields the clean Markov chain and a “lower-layer-acts-on-data” story, but flipping the order still feels mathematically valid; I want to better internalize why it’s less convenient for the CNN analogy.
* Coupling across parts: I get the theory for why a bag-of-words factorization is wrong; a minimal concrete counterexample would help intuition.

## 7

* Kernel size × SAL study (primary focus)
  * Goal: Understand how kernel size (and effective receptive field) shapes SAL’s local marginalization—i.e., the invariance–selectivity trade-off and implications for downstream transformations.
  * Actions:
    * Specify a small benchmark (synthetic or curated real images) with controlled small deformations (translation/rotation/scale jitter).
    * Implement 3 model variants with kernel sizes 3×3, 5×5, 7×7 (keep params comparable; same pooling).
    * Metrics: (i) stability of feature/logit responses under small $g$, (ii) accuracy on near-miss classes, (iii) sensitivity to occlusion.
    * Deliverables: plots of stability vs. kernel size; short write-up on how kernel size affects later layers’ ability to compose transformations.
* Residual vs. plain blocks under nuisance (planning + setup)
  * Goal: Test whether skip connections preserve selectivity that pooling/marginalization might wash out.
  * Actions:
    * Hypothesize the transformations that take place at the end of the residual block
    * Prepare matched configs (with/without skips) on the same nuisance-heavy setup from (A).
    * Define evaluation across unseen deformation magnitudes.
    * Deliverable: comparison table template + scripts ready to run.
* “Add-back” nuisances (design pass)
  * Goal: Determine a practical route to re-introduce nuisance parameters for approximate regeneration without sacrificing invariance in the main path.
  * Actions:
    * Look at how the connection from the end of the residual block preserves the nuisance variables
    * Use that knowledge to see how to reintroduce the nuisance variable
    * Test how regeneration works on the new scene distribution